# ARIMA++

The purpose of this notebook is to run ARIMA, etc.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sn
from pathlib import Path
from sklearn.model_selection import TimeSeriesSplit
from extract_and_clean import Clean

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error as mse

#now more time-series specific things
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import statsmodels.tsa.api as sm
import pmdarima as pm
from pmdarima import auto_arima
import itertools
from statsmodels.tsa.statespace.sarimax import SARIMAX

system_ids = [2105, 2107, 7333, 9068] #the relevant ids
#system_id = 2105 #may have to look through them eventually
#path = f'../../../../data_ds_project/systems/prize/{system_id}/'

def systemPath(system_id):
    return f'../../../../data_ds_project/systems/prize/{system_id}/'

#Instead of using all the data/files for EDA, we will use only some
#we begin with a list of inverters, and then add on the meters, to make sure we're choosing from all of the possibilities
individual_data = pd.read_csv('../../../../data_ds_project/systems/prize/new_inverter_names_for_prize_cleaned_power.csv')
individual_data

meters_df = pd.DataFrame({
    'system_id' : [2105, 9068], 
    'old_name' : ['meter','meter'], 
    'new_name' : ['000','000']
})
all_datas = pd.concat([individual_data, meters_df], ignore_index = True)
for i in range(len(all_datas)):
    all_datas.iloc[i,2] = str(all_datas['new_name'][i]).zfill(3)
all_datas['file_name'] = (all_datas['system_id'].astype(str) + "_"
                          + np.where(all_datas['old_name'] == 'meter', 'meter', 'inverter') + "_"
                          + all_datas['new_name'].astype(str))
all_datas

sample_data = all_datas #update as needed

,system_id,old_name,new_name,file_name
0,2107,inv_01_ac_power_inv_149583,000,2107_inverter_000
1,2107,inv_02_ac_power_inv_149588,001,2107_inverter_001
2,2107,inv_03_ac_power_inv_149593,002,2107_inverter_002
3,2107,inv_04_ac_power_inv_149598,003,2107_inverter_003
4,2107,inv_05_ac_power_inv_149603,004,2107_inverter_004
...,...,...,...,...
143,9068,inverter_module_2.4_ac_power_(kw)_inv_150142,007,9068_inverter_007
144,9068,inverter_1_ac_power_(kw)_inv_150143,008,9068_inverter_008
145,9068,inverter_2_ac_power_(kw)_inv_150144,009,9068_inverter_009
146,2105,meter,000,2105_meter_000


We need to figure out the optimal ARIMA constants. We know that seasonal = 365. We need to figre out (p,q,d) and (P,Q,D).

In [ ]:
#First, run straight auto-ARIMA. No filling in the missing data points. This will be a baseline for using ARIMA.
# Why? Missing data shouldn't mess with *too* much -- only with appx as many data points as there are chunks missing

#will write this to run through ALL of the systems, though to test the code, it'll only run one.
sample_data = all_datas
all_errors = np.zeros(len(sample_data))
for i in range(len(sample_data)):
    #file name
    file_name = all_datas.loc[i, 'file_name'] + ".csv"
    print(f"Now working on {file_name}")

    df = pd.read_csv(f'../../../../data_ds_project/systems/prize/prize_cleaned_power/{file_name}')
    #print(f"System_id = {str(sample_data.iloc[i,0])}")

    #turn strings into daytime
    df['time'] = pd.to_datetime(df['time'])

    max = df['power'].max()
    df['proportion'] = df['power']/max

    #train test split
    df_train = df.loc[:int(0.8*len(df))]
    df_test = df.loc[int(0.8*len(df)):]
    x_train = df_train['proportion']

    auto_model = auto_arima(df['proportion'], display = False)
    order = auto_model.order

    #now that we have the model, should do cross-validation
    n_val = 3
    errors_sys = np.zeros(n_val)
    kfold = TimeSeriesSplit(n_val)
    for j,(ind_train, ind_ho) in enumerate(kfold.split(x_train)):
        train = x_train.iloc[ind_train]
        ho = x_train.iloc[ind_ho]

        model = SARIMAX(
            train,
            order=order,
            enforce_stationarity=False,
            enforce_invertibility=False
        )

        mod = model.fit()
        y_pred = mod.forecast(steps = len(ho))
        
        errors_sys[j] = mse(y_pred, ho)
    all_errors[i] = errors_sys.mean()  

print(all_errors)


Now working on 2107_inverter_000.csv
Now working on 2107_inverter_001.csv
Now working on 2107_inverter_002.csv
Now working on 2107_inverter_003.csv
Now working on 2107_inverter_004.csv
Now working on 2107_inverter_005.csv
Now working on 2107_inverter_006.csv
Now working on 2107_inverter_007.csv
Now working on 2107_inverter_008.csv
Now working on 2107_inverter_009.csv
Now working on 2107_inverter_010.csv


c:\Users\hjpha\anaconda3\envs\erdos_ds_environment\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


Now working on 2107_inverter_011.csv
Now working on 2107_inverter_012.csv
Now working on 2107_inverter_013.csv
Now working on 2107_inverter_014.csv
Now working on 2107_inverter_015.csv
Now working on 2107_inverter_016.csv
Now working on 2107_inverter_017.csv
Now working on 2107_inverter_018.csv
Now working on 2107_inverter_019.csv
Now working on 2107_inverter_020.csv
Now working on 2107_inverter_021.csv
Now working on 2107_inverter_022.csv
Now working on 2107_inverter_023.csv
Now working on 7333_inverter_000.csv
Now working on 7333_inverter_001.csv
Now working on 7333_inverter_002.csv
Now working on 7333_inverter_003.csv
Now working on 7333_inverter_004.csv
Now working on 7333_inverter_005.csv
Now working on 7333_inverter_006.csv
Now working on 7333_inverter_007.csv
Now working on 7333_inverter_008.csv
Now working on 7333_inverter_009.csv
Now working on 7333_inverter_010.csv
Now working on 7333_inverter_011.csv
Now working on 7333_inverter_012.csv
Now working on 7333_inverter_013.csv
N

c:\Users\hjpha\anaconda3\envs\erdos_ds_environment\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


Now working on 7333_inverter_015.csv
Now working on 7333_inverter_016.csv
Now working on 7333_inverter_017.csv
Now working on 7333_inverter_018.csv
Now working on 7333_inverter_019.csv
Now working on 7333_inverter_020.csv


...

Now working on 7333_inverter_110.csv
Now working on 7333_inverter_111.csv
Now working on 9068_inverter_000.csv
Now working on 9068_inverter_001.csv
Now working on 9068_inverter_002.csv
Now working on 9068_inverter_003.csv
Now working on 9068_inverter_004.csv


c:\Users\hjpha\anaconda3\envs\erdos_ds_environment\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


Now working on 9068_inverter_005.csv
Now working on 9068_inverter_006.csv
Now working on 9068_inverter_007.csv
Now working on 9068_inverter_008.csv
Now working on 9068_inverter_009.csv
Now working on 2105_meter_000.csv


c:\Users\hjpha\anaconda3\envs\erdos_ds_environment\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


Now working on 9068_meter_000.csv
[0.03084541 0.04143531 0.04017069 0.06257677 0.03531425 0.04117115
 0.05583109 0.044206   0.03171434 0.0353297  0.05514185 0.03249624
 0.03209275 0.05682044 0.05430926 0.04916535 0.0316936  0.06083445
 0.0342492  0.03308522 0.03600134 0.02816212 0.02705038 0.03226479
 0.0832664  0.08385432 0.08200237 0.08341671 0.09682007 0.0726852
 0.05446444 0.07834931 0.08252    0.07643795 0.0815462  0.11243436
 0.08160965 0.08369144 0.07399196 0.06856302 0.07945604 0.08374219
 0.09152702 0.08187033 0.0773218  0.10063922 0.06536981 0.08602751
 0.07914924 0.07758162 0.07827464 0.07025886 0.04995794 0.08548062
 0.07062754 0.04237859 0.07766799 0.06454855 0.07374055 0.07296082
 0.05163299 0.09288471 0.0671612  0.09474915 0.09080205 0.08134936
 0.09316276 0.07258743 0.0937105  0.07790756 0.08950415 0.06985204
 0.11492161 0.09975337 0.08892818 0.08369922 0.09386795 0.07265449
 0.08498451 0.08779722 0.08896042 0.08034771 0.09485173 0.08745681
 0.07366543 0.08074899 0.0757

In [47]:
#save these errors *just in case* -- won't have to run the model again
naive_arima_errors = pd.DataFrame(all_errors)
naive_arima_errors.to_csv('errors/naive_arima_errors.csv')

Now we become less naive. In particular, quick plots of daily average data indicate that there is *some* seasonality. The season appears to be 1 year -- 365 days. However, this is an incredibly long season. 

In [31]:
#run ONE SARIMA -- see how long it takes
i=0
file_name = str(sample_data.iloc[i,0])+"_"
if sample_data.iloc[i,1]=="meter":
    file_name = file_name + "meter_"
else:
    file_name = file_name + "inverter_"
file_name = file_name + str(sample_data.iloc[i,2]).zfill(3) + ".csv"

df = pd.read_csv(f'../../../../data_ds_project/systems/prize/prize_cleaned_power/{file_name}')
    #print(f"System_id = {str(sample_data.iloc[i,0])}")

    #turn strings into daytime
df['time'] = pd.to_datetime(df['time'])

max = df['power'].max()
df['proportion'] = df['power']/max

    #fill in empty times 
obj = Clean()
new_df = obj.fill_missing_days_na(df)
    #display(new_df)

    #should do a train/test split
new_df_train = new_df.loc[:0.8*int(len(new_df))]

y=new_df_train['proportion']

check_stationarity(df['proportion'])

#do cross validation?
n_split=3
kfold = TimeSeriesSplit(n_split)
errors = np.zeros(n_split)
# for j,(train_ind,ho_ind) in enumerate(kfold.split(y)):
#     train = y[train_ind]
#     ho = y[ho_ind]

#     t = np.arange(len(train))

#     X = np.column_stack([
#         np.sin(2*np.pi*t/365),
#         np.cos(2*np.pi*t/365),
#     ])
#     model = sm.ARIMA(train, exog=X, order=(2,0,1)).fit()

#     t_future = np.arange(len(train), len(train) + len(ho))
#     X_future = np.column_stack([
#         np.sin(2*np.pi*t_future/365),
#         np.cos(2*np.pi*t_future/365),
#     ])
#     y_pred = model.forecast(steps = len(ho), exog = X_future)
    
#     display(new_df_train['time'][-len(y_pred):])
#     #create plot
#     plt.plot(new_df_train['time'], y, label = "True values")
#     plt.plot(new_df_train['time'][-len(y_pred):], y_pred, label = "Predicted values", color= "red")
#     plt.legend()
#     plt.title(f"Fold #{j}")
#     plt.show()
#     #errors[j] = mse(y_pred, ho)
#print(errors)
# p = 2
# d = 0
# q = 1
# P = 0
# D = 1
# Q = 0
# s = 365
# model = sm.ARIMA(y, order = (p, d, q), seasonal_order=(P, D, Q, s)).fit()


ADF Statistic: -7.872728475909532
p-value: 4.935561788400497e-12
Stationary


In [ ]:
#check whether model is any good

